# Fine-tune Florence-2 on VNImage2Text Dataset

In [1]:
!pip install -q transformers accelerate datasets timm einops
!pip install -q wandb nltk rouge-score rouge pycocoevalcap

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 7.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive

# 1. Kết nối Colab với Google Drive
drive.mount("/content/drive")

# 2. Xóa sạch thư mục dataset cũ đang bị lỗi byte
!rm -rf /content/dataset

# 3. Giải nén file zip nguyên vẹn từ Google Drive
!unzip -q /content/drive/MyDrive/dataset.zip -d /content/

Mounted at /content/drive


## 1. Load Dataset & Split

In [3]:
from datasets import DatasetDict, load_from_disk

# Nạp trực tiếp thư mục arrow dataset đã lưu trên disk
loaded_data = load_from_disk("dataset")

# Nếu dữ liệu đã có sẵn split 'train', lấy ra split; nếu chưa thì chia trực tiếp
if isinstance(loaded_data, DatasetDict) and "train" in loaded_data:
    base_data = loaded_data["train"]
else:
    base_data = loaded_data

split_data = base_data.train_test_split(test_size=0.2, seed=42)

data = DatasetDict(
    {"train": split_data["train"], "validation": split_data["test"]}
)

print(data["train"])
print(data["validation"])

Dataset({
    features: ['image', 'title', 'question', 'description'],
    num_rows: 1428
})
Dataset({
    features: ['image', 'title', 'question', 'description'],
    num_rows: 358
})


## 2. Load Florence-2 Model & Processor

In [4]:
!pip install -U "transformers==4.46.3" "huggingface-hub>=0.25.0,<1.0.0" accelerate timm einops

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 43.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.28.0
    Uninstalling huggingface_hub-1.28.0:
      Successfully uninstalled huggingface_hub-1.28.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled trans

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "microsoft/Florence-2-base-ft"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float32,  # Force float32 instead of float16
).to(device)

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/2.43k [00:00<?, ?B/s]

configuration_florence2.py:   0%|          | 0.00/15.1k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base-ft:
- configuration_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_florence2.py:   0%|          | 0.00/127k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base-ft:
- modeling_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


OSError: Can't load the model for 'microsoft/Florence-2-base-ft'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'microsoft/Florence-2-base-ft' is the correct path to a directory containing a file named pytorch_model.bin, tf_model.h5, model.ckpt or flax_model.msgpack.

## 3. Create PyTorch Dataset

In [ ]:
import random
import json
from torch.utils.data import Dataset

class CustomVQADataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]
        image = example['image']
        
        if image.mode != "RGB":
            image = image.convert("RGB")
            
        # Lấy dữ liệu hội thoại từ column 'conversations'
        raw_convs = example['conversations']
        
        # Đề phòng trường hợp thư viện datasets đọc mảng JSON thành định dạng string
        if isinstance(raw_convs, str):
            convs = json.loads(raw_convs)
        else:
            convs = raw_convs
            
        # Gom nhóm các cặp User (Question) - Assistant (Answer) liền kề
        qa_pairs = []
        for i in range(len(convs) - 1):
            if convs[i]['role'] == 'user' and convs[i+1]['role'] == 'assistant':
                qa_pairs.append((convs[i]['content'], convs[i+1]['content']))
                
        # Lấy ngẫu nhiên 1 cặp cho epoch hiện tại
        if len(qa_pairs) > 0:
            question, answer = random.choice(qa_pairs)
        else:
            # Fallback an toàn nếu data lỗi
            question, answer = "", "" 

        # Florence-2 thường cần task prefix <vqa>
        prompt = "<vqa> " + question if "<vqa>" not in question.lower() else question
        
        return prompt, answer, image

## 4. DataLoader

In [ ]:
import os
from torch.utils.data import DataLoader
from tqdm import tqdm

def collate_fn(batch):
    questions, answers, images = zip(*batch)
    inputs = processor(text=list(questions), images=list(images), return_tensors="pt", padding=True).to(device)
    return inputs, answers

train_dataset = CustomVQADataset(data['train'])
val_dataset = CustomVQADataset(data['validation'])

batch_size = 4
num_workers = 0

train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=collate_fn, num_workers=num_workers, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, collate_fn=collate_fn, num_workers=num_workers)

## 5. Fine-tuning and Evaluation loop

In [ ]:
import wandb
from transformers import AdamW, get_scheduler
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge import Rouge
from pycocoevalcap.cider.cider import Cider
import nltk

nltk.download('punkt')

def train_model(train_loader, val_loader, model, processor, epochs=3):
    optimizer = AdamW(model.parameters(), lr=1e-5)
    num_training_steps = epochs * len(train_loader)
    lr_scheduler = get_scheduler(
        name="linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )

    rouge = Rouge()
    cider = Cider()

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        predictions = []
        references = []

        for batch in tqdm(train_loader, desc=f"Training Epoch {epoch + 1}/{epochs}"):
            inputs, answers = batch
            input_ids = inputs["input_ids"].to(device)
            pixel_values = inputs["pixel_values"].to(device)
            labels = processor.tokenizer(text=answers, return_tensors="pt", padding=True, return_token_type_ids=False).input_ids.to(device)

            outputs = model(input_ids=input_ids, pixel_values=pixel_values, labels=labels)
            loss = outputs.loss

            loss.backward()
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

            train_loss += loss.item()

            pred_ids = torch.argmax(outputs.logits, dim=-1)
            predictions.append(pred_ids.cpu().numpy())
            references.append(labels.cpu().numpy())

        avg_train_loss = train_loss / len(train_loader)
        print(f"Average Training Loss: {avg_train_loss}")
        wandb.log({"Training Loss": avg_train_loss, "Epoch": epoch + 1})

        # Validation
        model.eval()
        val_loss = 0
        val_predictions = []
        val_references = []

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Validation Epoch {epoch + 1}/{epochs}"):
                inputs, answers = batch
                input_ids = inputs["input_ids"].to(device)
                pixel_values = inputs["pixel_values"].to(device)
                labels = processor.tokenizer(text=answers, return_tensors="pt", padding=True, return_token_type_ids=False).input_ids.to(device)

                outputs = model(input_ids=input_ids, pixel_values=pixel_values, labels=labels)
                loss = outputs.loss
                val_loss += loss.item()

                pred_ids = torch.argmax(outputs.logits, dim=-1)
                val_predictions.append(pred_ids.cpu().numpy())
                val_references.append(labels.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        print(f"Average Validation Loss: {avg_val_loss}")
        wandb.log({"Validation Loss": avg_val_loss, "Epoch": epoch + 1})

        # Eval Metrics on Validation
        flat_val_predictions = [pred.flatten() for sublist in val_predictions for pred in sublist]
        flat_val_references = [ref.flatten() for sublist in val_references for ref in sublist]

        decoded_val_predictions = [processor.decode(pred, skip_special_tokens=True) for pred in flat_val_predictions]
        decoded_val_references = [processor.decode(ref, skip_special_tokens=True) for ref in flat_val_references]

        tokenized_val_references = [[ref.split()] for ref in decoded_val_references]
        tokenized_val_predictions = [pred.split() for pred in decoded_val_predictions]

        smoothing_function = SmoothingFunction().method1
        bleu_val_score = corpus_bleu(tokenized_val_references, tokenized_val_predictions, smoothing_function=smoothing_function)
        wandb.log({"BLEU Score (Validation)": bleu_val_score})

        val_gts = {}
        val_res = {}
        for i in range(len(decoded_val_references)):
            img_id = f'img_{i}'
            val_gts[img_id] = [decoded_val_references[i]]
            val_res[img_id] = [decoded_val_predictions[i]]

        cider_val_score, _ = cider.compute_score(val_gts, val_res)
        wandb.log({"CIDEr Score (Validation)": cider_val_score})

        try:
            rouge_val_scores = rouge.get_scores(decoded_val_predictions, decoded_val_references, avg=True)
            wandb.log({
                "ROUGE-1 (Validation)": rouge_val_scores['rouge-1']['f'],
                "ROUGE-2 (Validation)": rouge_val_scores['rouge-2']['f'],
                "ROUGE-L (Validation)": rouge_val_scores['rouge-l']['f']
            })
        except:
            pass

        output_dir = f"./model_checkpoints/epoch_{epoch + 1}"
        os.makedirs(output_dir, exist_ok=True)
        model.save_pretrained(output_dir)
        processor.save_pretrained(output_dir)

    wandb.finish()

## 6. Run Training

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("Đang chạy bằng CPU!")

In [ ]:
import os
from google.colab import userdata
import wandb

# 1. Set API key from Colab Secrets before initializing
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

# 2. Initialize a single W&B run
wandb.init(
    project="florence2-finetune",
    name="florence2-frozen-vision",
    config={
        "epochs": 3,
        "freeze_vision_tower": True,
    },
)

# 3. Freeze vision tower parameters
for param in model.vision_tower.parameters():
    param.requires_grad = False

# 4. Train
train_model(train_loader, val_loader, model, processor, epochs=3)

# 5. Cleanly finish the W&B run
wandb.finish()

## 7. Inference on Custom Images in `test_images/`

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

def run_inference(image_path, model, processor):
    image = Image.open(image_path)
    if image.mode != "RGB":
        image = image.convert("RGB")

    task_prompt = "<vqa> What is the name of the tourist destination shown in this image?"
    inputs = processor(text=task_prompt, images=image, return_tensors="pt").to(device)

    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3
    )
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed_answer = processor.post_process_generation(generated_text, task=task_prompt, image_size=(image.width, image.height))

    plt.imshow(image)
    plt.axis("off")
    plt.show()
    print("Prediction:", parsed_answer)

test_images_dir = "test_images"
if os.path.exists(test_images_dir):
    for img_name in os.listdir(test_images_dir):
        img_path = os.path.join(test_images_dir, img_name)
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            print(f"--- Inferencing on {img_name} ---")
            run_inference(img_path, model, processor)
else:
    print(f"Directory {test_images_dir} does not exist.")